# Dokumente laden

Alle PDF dokumente aus dem `data` Ordner werden geladen. Der Inhalt wird dabei vom restlichen Text getrennt. Für den [OpenDataLoader](https://github.com/opendataloader-project/opendataloader-pdf) muss Java installiert sein.

In [ ]:
from langchain_opendataloader_pdf import OpenDataLoaderPDFLoader
import glob

loader = OpenDataLoaderPDFLoader(
    file_path=glob.glob("data/*.pdf"),
    format="markdown"
)
docs = loader.load()
for doc in docs:
    print(doc.page_content)


# Indexing
Der Inhalt aus den Dokumenten wird in Abschnitte unterteilt. Diese Abschnitte werden in hochdimensionale Vektoren kodiert, sodass der Text unabhängig von bestimmten Schlagwörtern nach relevanten Passagen zu einer Frage durchsucht werden kann.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()  # Load API keys

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

vectorstore = InMemoryVectorStore.from_documents(
    documents=all_splits,
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever(k=6)

# LLM und Prompt-Generierung

Aus dem Vektor Store werden zu jedem Prompt relevante Textstellen gefunden und angehängt, damit das LLM anhand dieser Textstellen die Frage beantworten kann. Der Prompt aus Frage und Textstellen wird dann an einen LLM Provider gesendet. Die Antwort des LLMs wird zusammen mit den gefundenen Textstellen zurückgegeben.

In [ ]:
from langsmith import traceable
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-mini", temperature=1)

# Add decorator so this function is traced in LangSmith
@traceable()
def rag_bot(question: str) -> dict:
    # LangChain retriever will be automatically traced
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
       Use the following source documents to answer the user's questions.
       If you don't know the answer, just say that you don't know.
       Use three sentences maximum and keep the answer concise.

<context>
{docs_string}
</context>"""
    # langchain ChatModel will be automatically traced
    ai_msg = llm.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )
    return {"answer": ai_msg.content, "documents": docs}

# Test

Das RAG wird mit einem Datensatz aus Fragen getestet. Ein LLM beurteilt die Qualität der Antworten auf Basis der Musterantworten.

In [ ]:
from langsmith import Client
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI

client = Client()

examples = [
    {
        "inputs": {"question": "Gib mir die Kontaktinformationen der Studiengangsberatung."},
        "outputs": {
            "answer": "Die Studiengangsberatung ist Prof. Dr. Beate Rhein. Ihr Büro ist in Raum ZN-06-05. Sie ist telefonisch unter der Nummer 0221/8275-2291 und über die Mailadresse studiengangleitung-batin@th-koeln.de erreichbar."},
    },
    {
        "inputs": {"question": "Brauche ich ein eigenes Laptop? Welche Anforderungen muss es erfüllen?"},
        "outputs": {"answer": "Ein aktuelles Laptop mit 4-8 GB Arbeitsspeicher und Windows oder Linux wird empfohlen."},
    },
    {
        "inputs": {"question": "Wann finden die Prüfungen statt?"},
        "outputs": {"answer": "Die Prüfungen finden in der Prüfungszeit vom 9. Februar bis zum 13. März 2026 statt."},
    },
    {
        "inputs": {"question": "Wie viele Wochenstunden sollte ich für das Modul Mathematik 2 einplanen?"},
        "outputs": {"answer": "Das Modul Mathematik 2 hat 8 Semesterwochenstunden."},
    },
    {
        "inputs": {"question": "Wie viele Versuche habe ich, um eine Prüfung zu bestehen"},
        "outputs": {"answer": "Zu jedem Modul haben Sie 3 Prüfungsversuche. Ausschließlich an der TH Köln bekommen Studierende vier zusätzliche Prüfungsversuche für das gesamte Studium."},
    },
]

dataset_name = "MLWR RAG"
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(
        dataset_id=dataset.id,
        examples=examples
    )

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are a teacher grading a quiz. You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. (2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

grader_llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0).with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""
    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness],
    experiment_prefix="mlwr-rag-correctness",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)

# Explore results locally as a dataframe if you have pandas installed
# experiment_results.to_pandas()